# Reading

In [239]:
import pandas as pd
import os

In [240]:
parent_path = "C:/Users/quintyn_come/Codex/AIBT102/"

In [241]:
commerces_paris = pd.read_csv(os.path.join(parent_path, "scraping_and_preprocessing/processed_resources/commerces/commerces_osm_paris.csv"))
commerces_haute_garonne = pd.read_csv(os.path.join(parent_path, "scraping_and_preprocessing/processed_resources/commerces/commerces_osm_Haute-Garonne.csv"))

In [242]:
list_of_all_dfs = [
    commerces_paris,
    commerces_haute_garonne,
]

In [243]:
all_commerces = pd.concat(list_of_all_dfs, ignore_index=True)

In [244]:
all_commerces.head(5)

,source,name,activity_query,category,category_type,address,lat,lon,opening_hours,phone,website,siret,osm_type,osm_id,osm_url,iris_code,iris,iris_name,commune_name,iris_type
0,OpenStreetMap,Relais Total Du Pont Aval,all_commercial_establishments_paris,convenience,shop,75015,48.834677,2.266146,24/7,+33 1 53 78 05 01,https://store.totalenergies.fr/en_EN/NF078037,NaN,node,25213384,https://www.openstreetmap.org/node/25213384,751156001,6001,Javel 1,Paris 15e Arrondissement,H
1,OpenStreetMap,Relais des Chaufourniers,all_commercial_establishments_paris,convenience;gas,shop,75019,48.880181,2.370744,24/7,NaN,NaN,NaN,node,25214653,https://www.openstreetmap.org/node/25214653,751197301,7301,Villette 1,Paris 19e Arrondissement,H
2,OpenStreetMap,Total,all_commercial_establishments_paris,convenience;gas,shop,NaN,48.831580,2.315822,24/7,NaN,NaN,NaN,node,66492656,https://www.openstreetmap.org/node/66492656,751145616,5616,Plaisance 16,Paris 14e Arrondissement,H
3,OpenStreetMap,Haltes-garderie municipale,all_commercial_establishments_paris,kindergarten,amenity,NaN,48.831463,2.316597,NaN,NaN,https://mairie14.paris.fr/pages/creches-du-14e...,NaN,node,158268042,https://www.openstreetmap.org/node/158268042,751145616,5616,Plaisance 16,Paris 14e Arrondissement,H
4,OpenStreetMap,Moulin Créole,all_commercial_establishments_paris,restaurant,amenity,NaN,48.829834,2.322899,07:00-23:30,+33 1 45 39 31 31,https://www.moulin-creole-paris.fr,NaN,node,158603712,https://www.openstreetmap.org/node/158603712,751145613,5613,Plaisance 13,Paris 14e Arrondissement,H


Let's drop the columns that aren't relevant for our project:

In [245]:
all_commerces.drop(columns=["source", "activity_query", "address", "website", "siret", "osm_type", "osm_url", "iris", "iris_name", "commune_name", "iris_type", "phone"], inplace=True)

C:\Users\quintyn_come\AppData\Local\Temp\ipykernel_43968\1261644974.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  all_commerces.drop(columns=["source", "activity_query", "address", "website", "siret", "osm_type", "osm_url", "iris", "iris_name", "commune_name", "iris_type", "phone"], inplace=True)


# Cleaning

Let's check the number of NaN values:

In [246]:
all_commerces.isna().sum()

name              4534
category             0
category_type        0
lat                  0
lon                  0
opening_hours    40168
osm_id               0
iris_code            0
dtype: int64

In [247]:
print('Number of rows:', all_commerces.shape[0])

Number of rows: 74412


It seems that more that half opening hours are missing, we can't really replace them so we will drop them but for a better study it would be interesting to keep them.
Also some shops don't have a name I will drop those rows.

In [248]:
all_commerces.drop(columns=["opening_hours"], inplace=True)
all_commerces = all_commerces.dropna(subset=["name"])

C:\Users\quintyn_come\AppData\Local\Temp\ipykernel_43968\3449023489.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  all_commerces.drop(columns=["opening_hours"], inplace=True)


Let's check the number of NaN now:

In [249]:
all_commerces.isna().sum()

name             0
category         0
category_type    0
lat              0
lon              0
osm_id           0
iris_code        0
dtype: int64

In [250]:
print('Number of rows:', all_commerces.shape[0])

Number of rows: 69878


Great there aren't any NaN anymore.

# Category

In [251]:
all_commerces.head(5)

,name,category,category_type,lat,lon,osm_id,iris_code
0,Relais Total Du Pont Aval,convenience,shop,48.834677,2.266146,25213384,751156001
1,Relais des Chaufourniers,convenience;gas,shop,48.880181,2.370744,25214653,751197301
2,Total,convenience;gas,shop,48.831580,2.315822,66492656,751145616
3,Haltes-garderie municipale,kindergarten,amenity,48.831463,2.316597,158268042,751145616
4,Moulin Créole,restaurant,amenity,48.829834,2.322899,158603712,751145613


Let's check the number of categories and category types in our project:

In [252]:
print("Number of different category types:", len(all_commerces["category_type"].unique()))

Number of different category types: 4


In [253]:
print("Number of different categories:", len(all_commerces["category"].unique()))

Number of different categories: 516


It's seems that there are many categories but not many category types. Futhermore, one shop can be associated with multiple categories. To simplify we will regroup the categories in more unique and precise category type.

In [254]:
GROUPS = {
    "food_retail": {
        "bakery", "pastry", "butcher", "seafood", "greengrocer",
        "grocery", "supermarket", "convenience", "frozen_food",
        "cheese", "dairy", "deli", "confectionery", "chocolate",
        "honey", "beverages", "food", "pasta", "olive_oil",
        "spices", "health_food", "farm", "apiary", "beekeeper",
        "farmers_market", "flour_mill", "vegan_cheese", "drinks"
    },

    "food_and_drink": {
        "restaurant", "fast_food", "bar", "pub", "cafe",
        "coffee", "tea", "ice_cream", "pancake", "caterer",
        "brewery", "winery", "distillery", "alcohol", "wine",
        "beer", "breakfast", "brunch", "lunch"
    },

    "fashion_and_accessories": {
        "clothes", "shoes", "bag", "fashion_accessories",
        "boutique", "tailor", "dressmaker", "haberdashery",
        "fabric", "sewing", "fur", "leather", "watches",
        "jewelry", "jeweller", "goldsmith", "perfumery",
        "cosmetics", "sunglasses", "umbrella", "wigs",
        "accessories", "gloves", "canes", "razor"
    },

    "beauty_and_wellness": {
        "hairdresser", "beauty", "massage", "tattoo",
        "piercing", "therapist", "personal_service",
        "alternative_medicine", "herbalist", "psychic",
        "esoteric"
    },

    "healthcare": {
        "pharmacy", "chemist", "doctors", "physician",
        "dentist", "clinic", "healthcare", "laboratory",
        "medical_supply", "hearing_aids", "optician",
        "orthopedics", "veterinary", "nursing_service",
        "ambulance", "dental_technician", "medical_expertise"
    },

    "home_and_decoration": {
        "furniture", "interior_decoration", "interior_decorator",
        "interior_design", "kitchen", "houseware",
        "household_linen", "bed", "bathroom",
        "bathroom_furnishing", "lighting", "carpet",
        "flooring", "parquet", "curtain", "window_blind",
        "tableware", "kitchenware", "fireplace",
        "garden_centre", "upholsterer", "tiles"
    },

    "construction_and_hardware": {
        "doityourself", "hardware", "construction",
        "construction_company", "builder", "carpenter",
        "joiner", "plumber", "electrician", "hvac",
        "heating", "heating_engineer", "roofer", "mason",
        "stonemason", "painter", "tiler", "glaziery",
        "locksmith", "insulation", "renovations",
        "building_materials", "wood", "metal_works",
        "window_construction", "plasterer", "tinsmith",
        "floorer", "parquet_layer", "cabinet_maker",
        "chimney_sweeper"
    },

    "automotive_and_mobility": {
        "car", "car_repair", "car_parts", "car_wash",
        "car_rental", "car_sharing", "bicycle",
        "bicycle_repair", "motorcycle", "motorcycle_repair",
        "scooter", "electric_scooter", "fuel", "tyres",
        "mobility_scooter", "tractor", "gasvending=fuel"
    },

    "culture_leisure_and_sports": {
        "books", "records", "music", "video", "video_games",
        "toys", "games", "art", "antiques", "craft",
        "handicraft", "musical_instrument", "sports",
        "sports_equipment", "golf", "ski", "surf", "skate",
        "scuba_diving", "fishing", "hunting", "model",
        "anime", "dance", "art_school", "music_school"
    },

    "general_retail": {
        "variety_store", "department_store", "mall",
        "marketplace", "kiosk", "newsagent", "general",
        "country_store", "gift", "souvenir", "souvenirs",
        "second_hand", "pop_up_store", "popup", "collector",
        "lottery", "tobacco"
    },

    "professional_services": {
        "bank", "insurance", "financial", "financial_services",
        "financial_advisor", "investment_advisor",
        "accountant", "tax_advisor", "lawyer", "notary",
        "estate_agent", "property_management",
        "property_developer", "property_broker", "architect",
        "engineer", "consulting", "advertising_agency",
        "employment_agency", "surveyor", "developer"
    },

    "personal_and_repair_services": {
        "dry_cleaning", "laundry", "shoemaker", "shoe_repair",
        "electronics_repair", "repair", "printing", "printer",
        "copyshop", "photographer", "photo_studio",
        "key_cutter", "moving_company", "cleaning",
        "storage_rental", "courier", "callshop",
        "printery", "bookbinder", "sewing_supply",
        "sharpening", "lost_property"
    },

    "education_and_training": {
        "kindergarten", "childcare", "school", "university",
        "college", "educational_institution", "language_school",
        "tutoring", "cooking_school", "dancing_school"
    },

    "tourism_and_travel": {
        "travel_agency", "travel_agent", "rental",
        "guide", "visa", "ticket", "left_luggage"
    },

    "public_services_and_institutions": {
        "government", "police", "diplomatic", "administration",
        "administrative", "association", "ngo", "foundation",
        "charity", "union", "labor_union", "political_party",
        "community_centre", "social_centre", "social_facility",
        "international_organisation", "quango"
    },

    "agriculture_and_crafts": {
        "agrarian", "agricultural_engines", "gardener",
        "groundskeeping", "sawmill", "basket_maker",
        "pottery", "glassblower", "sculptor", "artist",
        "engraver", "luthier"
    },

    "energy_and_environment": {
        "energy", "energy_supplier", "water", "water_utility",
        "water_filter", "gas", "pest_control",
        "bureau_de_lutte_contre_les_nuisibles"
    }
}

# Build category-to-group mapping
CATEGORY_MAPPING = {
    category: group
    for group, categories in GROUPS.items()
    for category in categories
}


In [255]:
#To simplify we only associate the first group that matches in the list
def classifier(category):
    if pd.isna(category):
        return "unknown"
    for cat in str(category).lower().split(";"):
        cat = cat.strip()
        if cat in CATEGORY_MAPPING:
            return CATEGORY_MAPPING[cat]
    return "other"

all_commerces["category_group"] = all_commerces["category"].apply(classifier)

In [256]:
all_commerces

,name,category,category_type,lat,lon,osm_id,iris_code,category_group
0,Relais Total Du Pont Aval,convenience,shop,48.834677,2.266146,25213384,751156001,food_retail
1,Relais des Chaufourniers,convenience;gas,shop,48.880181,2.370744,25214653,751197301,food_retail
2,Total,convenience;gas,shop,48.831580,2.315822,66492656,751145616,food_retail
3,Haltes-garderie municipale,kindergarten,amenity,48.831463,2.316597,158268042,751145616,education_and_training
4,Moulin Créole,restaurant,amenity,48.829834,2.322899,158603712,751145613,food_and_drink
...,...,...,...,...,...,...,...,...
74407,La Galerie Espaces Fenouillet,mall,shop,43.690544,1.400681,8872294,311820000,general_retail
74408,Collins Aerospace,company,office,43.629169,1.380980,10049391,310690110,other
74409,Centre Commercial Blagnac,mall,shop,43.645543,1.372083,10162755,310690111,general_retail
74410,Centre d'Intérêt Public,mall,shop,43.517883,1.560674,19221272,311690101,general_retail


In [257]:
import os
output_dir = os.path.join(parent_path, "cleaning/cleaned_resources")
output_name = "all_commerces.csv"
output_path = os.path.join(output_dir, output_name)
os.makedirs(output_dir, exist_ok=True)
all_commerces.to_csv(
    output_path,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)